<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Backpropagation and autograd

**[Backpropagation and autograd](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/)** · Deep Learning and Transformers Explained · Module 3, lesson 3 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** follow one neuron forward and backward with real numbers, let PyTorch's autograd compute the same gradients, and check them by hand.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account, no graphics card. |
| **You should know** | The loss and gradient descent, from [Loss: how wrong is the network?](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/) and [Gradient descent](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M03-L03-backpropagation-and-autograd/backpropagation-and-autograd-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One neuron, forward

The one-neuron network of the last lesson, on ticket C: it says "refund" twice (x = 2) and it is a payment ticket (y = 1). Take w = 0.5 and b = −1.

`requires_grad=True` tells PyTorch: "this number is a parameter; keep a record of every calculation that uses it, so you can compute its gradient later". That record is the **computation graph**.

> **Predict.** Compute z = w·x + b, p = sigmoid(z) and the loss −ln p by hand first. Then run the cell.

In [ ]:
x, y = 2.0, 1.0                                # ticket C: "refund" twice, a payment ticket
w = torch.tensor(0.5, requires_grad=True)      # the weight: keep a record for its gradient
b = torch.tensor(-1.0, requires_grad=True)     # the bias: the same
z = w * x + b
p = torch.sigmoid(z)
loss = -torch.log(p)
print("z =", z)
print("p =", p)
print("loss =", loss)

> **Check.** You should see `z = tensor(0., grad_fn=<AddBackward0>)`, `p = tensor(0.5000, grad_fn=<SigmoidBackward0>)` and `loss = tensor(0.6931, grad_fn=<NegBackward0>)`.

Each `grad_fn` names the step that made the number: an addition, a sigmoid, a negation. Together they are the graph that the backward pass walks through.

## 2. One neuron, backward

The **chain rule** passes the blame back one step at a time. Each step multiplies the blame that arrives by its own local slope:

- −ln p at p = 0.5: local slope −1/p = −2. The loss changes −2 per unit of p.
- sigmoid at z = 0: local slope p(1 − p) = 0.25. So the loss changes −2 × 0.25 = −0.5 per unit of z.
- z = w·x + b: z changes x = 2 per unit of w, and 1 per unit of b. So grad w = −0.5 × 2 = −1.0 and grad b = −0.5 × 1 = −0.5.

> **Predict.** Write down grad w and grad b from the list above. Then run the cell: `loss.backward()` walks the graph back and stores each gradient in `.grad`.

In [ ]:
loss.backward()
print("grad w =", w.grad)
print("grad b =", b.grad)

> **Check.** You should see `grad w = tensor(-1.)` and `grad b = tensor(-0.5000)`: the numbers from the chain rule.

## 3. Check it by nudging

The last lesson measured slopes by nudging. Nudge w by 0.001 to each side and compare.

> **Predict.** Will the nudged slope agree with `w.grad` to three decimals? Then run the cell.

In [ ]:
def neuron_loss(w_value, b_value=-1.0):
    return -torch.log(torch.sigmoid(torch.tensor(w_value) * x + b_value)).item()

nudge = 1e-3
print("nudged slope for w:", (neuron_loss(0.5 + nudge) - neuron_loss(0.5 - nudge)) / (2 * nudge))

> **Check.** You should see a number very close to `-1.0` (for example `-0.99998…` or `-1.00001…`; the last digits depend on rounding).

Both ways agree. The difference is the cost: nudging needs two extra loss computations **per parameter**; backpropagation gets every gradient from one backward pass.

## 4. Gradients add up

`.grad` **adds** each new gradient to the one already there. This is useful when one parameter is used in several places, and it is the reason for the first step of the training loop in the next lesson.

> **Predict.** Run the forward pass and `backward()` again, without clearing `w.grad`. What will `w.grad` be? Then run the cell.

In [ ]:
loss = -torch.log(torch.sigmoid(w * x + b))   # the same forward pass again
loss.backward()
print("grad w after a second backward:", w.grad)
w.grad = None                                  # clear it, so the next cells start from zero
b.grad = None

> **Check.** You should see `tensor(-2.)`: −1 from the first backward pass plus −1 from the second.

## 5. The ticket network: check one gradient by hand

For softmax with cross-entropy, the gradient of each **score** has a simple form: the probability minus 1 for the right team, and the probability itself for every other team. The output bias of a team is added to that team's score, so its gradient is the same number.

> **Predict.** The untrained network gives the first validation ticket the probabilities 0.1722, 0.2003, 0.2180, 0.2006 and 0.2089, and its right team is payment (index 2). Write down the five gradients of the output bias. Then run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
scores = model(x_valid[:1])
loss = loss_fn(scores, y_valid[:1])
loss.backward()
print("probabilities:      ", torch.softmax(scores, dim=1).detach())
print("grad of output bias:", model.layers[3].bias.grad)

> **Check.** You should see the probabilities `[[0.1722, 0.2003, 0.2180, 0.2006, 0.2089]]` and the gradients `[ 0.1722,  0.2003, -0.7820,  0.2006,  0.2089]`. `.detach()` only hides the `grad_fn` note in the print.

The payment bias has a negative gradient, −0.782 = 0.218 − 1: raise it to lower the loss. Every other bias has a positive gradient: lower it. Gradient descent will make payment more likely for this ticket.

## 6. One gradient for every parameter

> **Predict.** What shape will the gradient of the first layer's weights have? Then run the cell.

In [ ]:
for name, weight in model.named_parameters():
    print(f"{name:16} weight {str(tuple(weight.shape)):12} grad {tuple(weight.grad.shape)}")

> **Check.** You should see four lines: `layers.0.weight` (64, 2864), `layers.0.bias` (64,), `layers.3.weight` (5, 64) and `layers.3.bias` (5,), each with a gradient of the same shape.

Every parameter gets a gradient of its own shape: 183,685 slopes from one backward pass.

## 7. Why not nudge? Time it

> **Predict.** One forward pass on all 4,696 training tickets takes a few hundredths of a second. How long would nudging all 183,685 parameters take, for one step? Then run the cell. Your times will differ from the lesson's.

In [ ]:
import time

start = time.perf_counter()
loss_fn(model(x_train), y_train)
forward = time.perf_counter() - start

start = time.perf_counter()
model.zero_grad()
loss_fn(model(x_train), y_train).backward()
both = time.perf_counter() - start

print(f"one forward pass:     {forward:.3f} s")
print(f"forward and backward: {both:.3f} s")
print(f"nudging every parameter once: about {forward * count_parameters(model) / 60:.0f} minutes")

> **Check.** You should see a forward pass of about 0.01 to 0.05 seconds, a forward and backward pass that takes about the same, and many minutes for nudging.

Backpropagation costs about one extra pass, however many parameters there are. Nudging costs one pass per parameter.

## 8. Failure case: no record, no gradient

Tomás makes the weight and the bias without `requires_grad=True`.

> **Predict.** What happens at `loss.backward()`? Then run the cell.

In [ ]:
w2 = torch.tensor(0.5)       # no requires_grad
b2 = torch.tensor(-1.0)
loss2 = -torch.log(torch.sigmoid(w2 * x + b2))
try:
    loss2.backward()
except RuntimeError as error:
    print("RuntimeError:", error)

> **Check.** You should see `RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn`.

PyTorch kept no record, so there is no graph to walk back. **Fix:** make parameters with `requires_grad=True`. The layers of `torch.nn`, such as `nn.Linear`, do this for you.

## 9. Check yourself

> **Your turn.** Ticket B says "refund" once (x = 1) and is a payment ticket (y = 1). With w = 0.5 and b = −1, use autograd to compute the gradient of its loss −ln p for w. Put it in `grad_w_b` as a Python number (`.item()`). Then run the check cell.

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
loss = -torch.log(torch.sigmoid(w * 1.0 + b))
loss.backward()
grad_w_b = w.grad.item()
print(grad_w_b)

In [ ]:
expect('grad_w_b', grad_w_b, -0.6225, tolerance=0.001)

## Next

You can now follow a calculation forward and back, read a gradient as "how much the loss changes per unit of this parameter", and let PyTorch compute every gradient with `backward()`. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/) has the computation graph with every number and the knowledge checks. The next lesson puts the pieces together: the training loop.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.